# ARIA: E-Commerce Retail Intelligence Platform (BigQuery AI)

An end-to-end e-commerce data engineering and AI analytics project built on **Google Cloud BigQuery** using the free public dataset **`bigquery-public-data.thelook_ecommerce`**.

### Project Highlights
1. **Schema Design & Validation Pipelines:** Relational BigQuery schema across `customers`, `products`, and `transactions` tables with multi-table `JOIN`s, aggregations, and automated `UNION ALL` data-quality validation.
2. **Demand Forecasting (`AI.FORECAST` / `TimesFM`):** Zero-shot deep-learning time-series forecasting pipeline powered by Google Research's pretrained Transformer foundation model (`TimesFM`) predicting 30-day category demand with confidence intervals.
3. **Inventory Risk & Product Performance Prediction (`LINEAR_REG` + `ML.PREDICT`):** SKU-level Days-of-Supply (DoS) stockout risk classification combined with a BigQuery ML regression model predicting 30-day SKU revenue.
4. **Semantic Discovery & Generative Merchandising (`ML.GENERATE_EMBEDDING`, `VECTOR_SEARCH`, `ML.GENERATE_TEXT`):** Real Vertex AI remote models in BigQuery generating vector embeddings (`text-embedding-004`) for cosine-similarity product search and automated restock action plans (`gemini-1.5-flash`).

In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from google.cloud import bigquery

# Configuration
PROJECT_ID = os.getenv("GOOGLE_CLOUD_PROJECT", "your-gcp-project-id")
DATASET_ID = "aria_retail_intelligence"
CONNECTION_ID = "us.vertex_ai_conn"  # BigQuery Cloud Resource Connection for Vertex AI
DATASET_REF = f"{PROJECT_ID}.{DATASET_ID}"

client = bigquery.Client(project=PROJECT_ID, location="US")

# Ensure target dataset exists
dataset = bigquery.Dataset(DATASET_REF)
dataset.location = "US"
client.create_dataset(dataset, exists_ok=True)
print(f"Connected to BigQuery dataset: {DATASET_REF}")

---
## Module 1: BigQuery Schema Design, SQL Pipelines & Data Validation
We ingest and transform real e-commerce records from `bigquery-public-data.thelook_ecommerce` (`users`, `products`, `orders`, `order_items`, `inventory_items`) into three analytics-ready tables:
- **`customers`**: Customer demographics joined with completed order history (`total_orders`, `lifetime_spend`).
- **`products`**: SKU catalog with cost, retail price, gross margin, and a standardized `content` text column for `ML.GENERATE_EMBEDDING`.
- **`transactions`**: Completed order line-items over the trailing 365 days joined with product cost and category metadata.

In [2]:
# 1.1 Create Customers Table
customers_sql = f"""
CREATE OR REPLACE TABLE `{DATASET_REF}.customers` AS
SELECT
    u.id AS customer_id,
    u.age,
    u.gender,
    u.country,
    u.state,
    u.traffic_source,
    DATE(u.created_at) AS signup_date,
    COUNT(DISTINCT oi.order_id) AS total_orders,
    ROUND(COALESCE(SUM(oi.sale_price), 0), 2) AS lifetime_spend
FROM `bigquery-public-data.thelook_ecommerce.users` u
LEFT JOIN `bigquery-public-data.thelook_ecommerce.order_items` oi
    ON u.id = oi.user_id
    AND oi.status NOT IN ('Cancelled', 'Returned')
GROUP BY u.id, u.age, u.gender, u.country, u.state, u.traffic_source, DATE(u.created_at)
HAVING total_orders > 0
LIMIT 5000
"""
client.query(customers_sql).result()

# 1.2 Create Products Table (with `content` column for ML.GENERATE_EMBEDDING)
products_sql = f"""
CREATE OR REPLACE TABLE `{DATASET_REF}.products` AS
SELECT
    p.id AS product_id,
    p.name AS product_name,
    p.category,
    p.brand,
    p.department,
    ROUND(p.cost, 2) AS unit_cost,
    ROUND(p.retail_price, 2) AS retail_price,
    ROUND(p.retail_price - p.cost, 2) AS gross_margin,
    CONCAT(
        'Product: ', COALESCE(p.name, 'Unknown'),
        ' | Brand: ', COALESCE(p.brand, 'Generic'),
        ' | Category: ', COALESCE(p.category, 'General'),
        ' | Department: ', COALESCE(p.department, 'General'),
        ' | Price: $', CAST(ROUND(p.retail_price, 2) AS STRING)
    ) AS content
FROM `bigquery-public-data.thelook_ecommerce.products` p
WHERE p.name IS NOT NULL
  AND p.retail_price > 0
LIMIT 1000
"""
client.query(products_sql).result()

# 1.3 Create Transactions Table
transactions_sql = f"""
CREATE OR REPLACE TABLE `{DATASET_REF}.transactions` AS
SELECT
    oi.id AS transaction_id,
    oi.order_id,
    oi.user_id AS customer_id,
    oi.product_id,
    oi.inventory_item_id,
    oi.status AS order_status,
    DATE(oi.created_at) AS transaction_date,
    ROUND(oi.sale_price, 2) AS sale_price,
    ROUND(p.cost, 2) AS product_cost,
    ROUND(oi.sale_price - p.cost, 2) AS net_profit,
    p.category,
    p.brand
FROM `bigquery-public-data.thelook_ecommerce.order_items` oi
INNER JOIN `bigquery-public-data.thelook_ecommerce.products` p
    ON oi.product_id = p.id
WHERE oi.status NOT IN ('Cancelled', 'Returned')
  AND oi.created_at >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 365 DAY)
"""
client.query(transactions_sql).result()

# 1.4 Run Data Quality & Schema Validation Pipeline
validation_sql = f"""
SELECT
    'customers' AS table_name,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT customer_id) AS unique_primary_keys,
    COUNTIF(customer_id IS NULL) AS null_key_count,
    ROUND(AVG(lifetime_spend), 2) AS avg_metric_value
FROM `{DATASET_REF}.customers`
UNION ALL
SELECT
    'products' AS table_name,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT product_id) AS unique_primary_keys,
    COUNTIF(product_id IS NULL OR content IS NULL) AS null_key_count,
    ROUND(AVG(retail_price), 2) AS avg_metric_value
FROM `{DATASET_REF}.products`
UNION ALL
SELECT
    'transactions' AS table_name,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT transaction_id) AS unique_primary_keys,
    COUNTIF(transaction_id IS NULL OR sale_price <= 0) AS null_key_count,
    ROUND(AVG(sale_price), 2) AS avg_metric_value
FROM `{DATASET_REF}.transactions`
"""
df_validation = client.query(validation_sql).to_dataframe()
display(df_validation)

---
## Module 2: Demand Forecasting (`AI.FORECAST` with Google `TimesFM`)
Instead of hardcoded multipliers or manually tuning ARIMA parameters, we aggregate daily unit sales across the top product categories and run BigQuery's native **`AI.FORECAST`** function. Under the hood, `AI.FORECAST` invokes Google Research's **TimesFM** (Time-Series Foundation Model)—a pretrained decoder-only Transformer architecture—to project 30-day demand with 90% prediction intervals in a single SQL query.

In [3]:
# 2.1 Build Daily Category Demand Time-Series Table
daily_demand_sql = f"""
CREATE OR REPLACE TABLE `{DATASET_REF}.daily_category_demand` AS
SELECT
    transaction_date,
    category,
    COUNT(transaction_id) AS daily_units_sold,
    ROUND(SUM(sale_price), 2) AS daily_revenue
FROM `{DATASET_REF}.transactions`
WHERE category IN (
    SELECT category
    FROM `{DATASET_REF}.transactions`
    GROUP BY category
    ORDER BY COUNT(*) DESC
    LIMIT 6
)
GROUP BY transaction_date, category
"""
client.query(daily_demand_sql).result()

# 2.2 Forecast Next 30 Days with Native BigQuery AI.FORECAST (TimesFM)
forecast_sql = f"""
SELECT
    category,
    DATE(forecast_timestamp) AS forecast_date,
    ROUND(forecast_value, 1) AS predicted_units,
    ROUND(GREATEST(0, prediction_interval_lower_bound), 1) AS lower_bound_units,
    ROUND(prediction_interval_upper_bound, 1) AS upper_bound_units,
    confidence_level
FROM AI.FORECAST(
    TABLE `{DATASET_REF}.daily_category_demand`,
    data_col => 'daily_units_sold',
    timestamp_col => 'transaction_date',
    id_cols => ['category'],
    horizon => 30,
    confidence_level => 0.90
)
ORDER BY category, forecast_date
"""
df_forecast = client.query(forecast_sql).to_dataframe()
display(df_forecast.head(15))

---
## Module 3: Inventory Risk & Product-Performance Prediction (`LINEAR_REG` & `ML.PREDICT`)
1. **Inventory Risk Pipeline:** Joins `bigquery-public-data.thelook_ecommerce.inventory_items` (unsold vs. sold stock) with 30-day transaction velocity to calculate **Days of Supply (DoS)** and classify SKUs into actionable risk tiers (`CRITICAL: Stockout`, `HIGH RISK: < 14 Days Supply`, `MODERATE RISK`, `OVERSTOCK`, `HEALTHY`).
2. **Product-Performance Prediction:** Trains a BigQuery ML **`LINEAR_REG`** model (`product_performance_model`) on product attributes, margins, stock levels, and sales velocity, evaluates it via **`ML.EVALUATE`**, and predicts 30-day SKU revenue via **`ML.PREDICT`**.

In [4]:
# 3.1 Build Inventory Risk Analytics Table
inventory_risk_sql = f"""
CREATE OR REPLACE TABLE `{DATASET_REF}.inventory_risk_report` AS
WITH inventory_status AS (
    SELECT
        product_id,
        COUNTIF(sold_at IS NULL) AS available_stock,
        COUNTIF(sold_at IS NOT NULL) AS historical_sold_units
    FROM `bigquery-public-data.thelook_ecommerce.inventory_items`
    GROUP BY product_id
),
recent_velocity AS (
    SELECT
        product_id,
        COUNT(transaction_id) AS units_sold_30d,
        ROUND(COUNT(transaction_id) / 30.0, 2) AS daily_sales_velocity,
        ROUND(SUM(sale_price), 2) AS revenue_30d
    FROM `{DATASET_REF}.transactions`
    WHERE transaction_date >= DATE_SUB(CURRENT_DATE(), INTERVAL 30 DAY)
    GROUP BY product_id
)
SELECT
    p.product_id,
    p.product_name,
    p.category,
    p.brand,
    p.retail_price,
    COALESCE(i.available_stock, 0) AS available_stock,
    COALESCE(v.units_sold_30d, 0) AS units_sold_30d,
    COALESCE(v.daily_sales_velocity, 0) AS daily_sales_velocity,
    COALESCE(v.revenue_30d, 0) AS revenue_30d,
    ROUND(
        SAFE_DIVIDE(COALESCE(i.available_stock, 0), NULLIF(v.daily_sales_velocity, 0)),
        1
    ) AS days_of_supply,
    CASE
        WHEN COALESCE(i.available_stock, 0) = 0 AND COALESCE(v.units_sold_30d, 0) > 0
            THEN 'CRITICAL: Stockout'
        WHEN SAFE_DIVIDE(i.available_stock, NULLIF(v.daily_sales_velocity, 0)) < 14
            THEN 'HIGH RISK: < 14 Days Supply'
        WHEN SAFE_DIVIDE(i.available_stock, NULLIF(v.daily_sales_velocity, 0)) < 30
            THEN 'MODERATE RISK: 14-30 Days Supply'
        WHEN COALESCE(v.units_sold_30d, 0) = 0 AND COALESCE(i.available_stock, 0) > 20
            THEN 'OVERSTOCK: Slow Mover'
        ELSE 'HEALTHY'
    END AS inventory_risk_tier
FROM `{DATASET_REF}.products` p
LEFT JOIN inventory_status i ON p.product_id = i.product_id
LEFT JOIN recent_velocity v ON p.product_id = v.product_id
"""
client.query(inventory_risk_sql).result()

# 3.2 Train Product Performance Regression Model (BQML)
perf_model_sql = f"""
CREATE OR REPLACE MODEL `{DATASET_REF}.product_performance_model`
OPTIONS(
    model_type = 'LINEAR_REG',
    input_label_cols = ['revenue_30d'],
    ls_init_learn_rate = 0.15,
    max_iterations = 20
) AS
SELECT
    p.category,
    p.brand,
    p.department,
    p.retail_price,
    p.unit_cost,
    p.gross_margin,
    COALESCE(r.available_stock, 0) AS available_stock,
    COALESCE(r.units_sold_30d, 0) AS units_sold_30d,
    COALESCE(r.revenue_30d, 0) AS revenue_30d
FROM `{DATASET_REF}.products` p
INNER JOIN `{DATASET_REF}.inventory_risk_report` r
    USING (product_id)
WHERE r.revenue_30d > 0
"""
client.query(perf_model_sql).result()

# 3.3 Evaluate & Predict with ML.EVALUATE and ML.PREDICT
df_eval = client.query(
    f"SELECT ROUND(mean_absolute_error, 2) AS mae, ROUND(r2_score, 4) AS r2_score FROM ML.EVALUATE(MODEL `{DATASET_REF}.product_performance_model`)"
).to_dataframe()
print("Model Evaluation:")
display(df_eval)

predict_perf_sql = f"""
SELECT
    product_id,
    product_name,
    category,
    brand,
    retail_price,
    units_sold_30d,
    revenue_30d AS actual_revenue_30d,
    ROUND(predicted_revenue_30d, 2) AS predicted_revenue_30d,
    inventory_risk_tier
FROM ML.PREDICT(
    MODEL `{DATASET_REF}.product_performance_model`,
    (
        SELECT
            p.product_id, p.product_name, p.category, p.brand, p.department,
            p.retail_price, p.unit_cost, p.gross_margin,
            r.available_stock, r.units_sold_30d, r.revenue_30d, r.inventory_risk_tier
        FROM `{DATASET_REF}.products` p
        INNER JOIN `{DATASET_REF}.inventory_risk_report` r USING (product_id)
    )
)
ORDER BY predicted_revenue_30d DESC
LIMIT 15
"""
df_predictions = client.query(predict_perf_sql).to_dataframe()
display(df_predictions)

---
## Module 4: Semantic Search & Generative Merchandising (`ML.GENERATE_EMBEDDING`, `VECTOR_SEARCH`, `ML.GENERATE_TEXT`)
We register two remote Vertex AI foundation models directly inside BigQuery:
1. **`embedding_model` (`text-embedding-004`):** Called via **`ML.GENERATE_EMBEDDING`** to encode all products into 768-dimensional vectors, stored in `product_embeddings`, and queried via native **`VECTOR_SEARCH`** (`distance_type => 'COSINE'`).
2. **`gemini_model` (`gemini-1.5-flash`):** Called via **`ML.GENERATE_TEXT`** to synthesize natural-language merchandising and restock action plans for high-risk SKUs.

In [5]:
# 4.1 Register Vertex AI Remote Models in BigQuery
client.query(f"""
CREATE OR REPLACE MODEL `{DATASET_REF}.embedding_model`
REMOTE WITH CONNECTION `{PROJECT_ID}.{CONNECTION_ID}`
OPTIONS (ENDPOINT = 'text-embedding-004')
""").result()

client.query(f"""
CREATE OR REPLACE MODEL `{DATASET_REF}.gemini_model`
REMOTE WITH CONNECTION `{PROJECT_ID}.{CONNECTION_ID}`
OPTIONS (ENDPOINT = 'gemini-1.5-flash')
""").result()

# 4.2 Generate Product Embeddings Table via ML.GENERATE_EMBEDDING
create_embeddings_sql = f"""
CREATE OR REPLACE TABLE `{DATASET_REF}.product_embeddings` AS
SELECT
    product_id,
    product_name,
    category,
    brand,
    retail_price,
    content,
    ml_generate_embedding_result AS embedding
FROM ML.GENERATE_EMBEDDING(
    MODEL `{DATASET_REF}.embedding_model`,
    TABLE `{DATASET_REF}.products`,
    STRUCT(TRUE AS flatten_json_output, 'RETRIEVAL_DOCUMENT' AS task_type)
)
WHERE ARRAY_LENGTH(ml_generate_embedding_result) > 0
"""
client.query(create_embeddings_sql).result()

# 4.3 Semantic Product Discovery via Native VECTOR_SEARCH
search_text = "comfortable breathable athletic running jacket for outdoor workouts"
vector_search_sql = f"""
SELECT
    query.query_text,
    base.product_id,
    base.product_name,
    base.category,
    base.brand,
    base.retail_price,
    ROUND(1 - distance, 4) AS cosine_similarity
FROM VECTOR_SEARCH(
    TABLE `{DATASET_REF}.product_embeddings`,
    'embedding',
    (
        SELECT
            '{search_text}' AS query_text,
            ml_generate_embedding_result AS embedding
        FROM ML.GENERATE_EMBEDDING(
            MODEL `{DATASET_REF}.embedding_model`,
            (SELECT '{search_text}' AS content),
            STRUCT(TRUE AS flatten_json_output, 'RETRIEVAL_QUERY' AS task_type)
        )
    ),
    top_k => 5,
    distance_type => 'COSINE'
)
ORDER BY cosine_similarity DESC
"""
df_semantic_results = client.query(vector_search_sql).to_dataframe()
print("Semantic VECTOR_SEARCH Results:")
display(df_semantic_results)

# 4.4 Automated Restock & Merchandising Action Briefs via ML.GENERATE_TEXT
generate_text_sql = f"""
SELECT
    product_id,
    product_name,
    category,
    inventory_risk_tier,
    ml_generate_text_llm_result AS ai_merchandising_action
FROM ML.GENERATE_TEXT(
    MODEL `{DATASET_REF}.gemini_model`,
    (
        SELECT
            product_id,
            product_name,
            category,
            inventory_risk_tier,
            CONCAT(
                'You are an e-commerce retail analyst. Write a concise 2-sentence action plan for ',
                product_name, ' (Category: ', category, ', Price: $', CAST(retail_price AS STRING), '). ',
                'Current stock: ', CAST(available_stock AS STRING), ' units, ',
                '30-day sales: ', CAST(units_sold_30d AS STRING), ' units, ',
                'Risk Status: ', inventory_risk_tier, '.'
            ) AS prompt
        FROM `{DATASET_REF}.inventory_risk_report`
        WHERE inventory_risk_tier IN ('CRITICAL: Stockout', 'HIGH RISK: < 14 Days Supply')
        ORDER BY revenue_30d DESC
        LIMIT 5
    ),
    STRUCT(
        0.2 AS temperature,
        150 AS max_output_tokens,
        TRUE AS flatten_json_output
    )
)
"""
df_ai_briefs = client.query(generate_text_sql).to_dataframe()
print("AI-Generated Merchandising Action Plans:")
display(df_ai_briefs)